# Simple MapReduce Word Count

This notebook simulates MapReduce on one computer. The pipeline:

```
5 input splits -> MAP (parallel) -> COMBINE -> PARTITION -> SHUFFLE & SORT -> REDUCE -> 2 output files
```

Each book file is **one split**, and each split gets **one mapper**.

## 0. Imports and settings

- `re` splits text into words.
- `ThreadPoolExecutor` runs the mappers at the same time.
- `defaultdict` groups values by word during the shuffle.



In [1]:
import re
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

DATA_DIR = Path("../datasets/text-files")  
OUTPUT_DIR = Path("output")                  
NUM_REDUCERS = 2

## 1. Input: one file = one split

We list the five book files. Each file is handed to exactly one mapper.

In [2]:
splits = [DATA_DIR / f"book_0{i}.txt" for i in range(1, 6)]

for s in splits:
    assert s.exists(), f"Missing file: {s.resolve()}  (check DATA_DIR)"
    print("split:", s.name)

split: book_01.txt
split: book_02.txt
split: book_03.txt
split: book_04.txt
split: book_05.txt


## 2. Map: emit `(word, 1)` pairs

The mapper reads one split and emits a pair `(word, 1)` for **every** word it sees.


In [3]:
def tokenize(text):
    return re.findall(r"[^\W\d_]+", text.lower())

def mapper(text):
    return [(word, 1) for word in tokenize(text)]

print(mapper("Mohamed Belal trial and trial ?_!3 423 2"))

[('mohamed', 1), ('belal', 1), ('trial', 1), ('and', 1), ('trial', 1)]


## 3. Combine: local subtotals

Sending `("the", 1)` thousands of times is wasteful. The **combiner** runs on the mapper's side and
adds up the pairs *inside one split* first, so we send `("the", 3500)` instead.

In [4]:
def combiner(pairs):
    counts = {}
    for word, n in pairs:
        counts[word] = counts.get(word, 0) + n
    return counts

print(combiner(mapper("The cat saw the dog.")))

{'the': 2, 'cat': 1, 'saw': 1, 'dog': 1}


## 4. Run the mappers at the same time

We start **one mapper per split** in a thread pool, so all five run concurrently.
Each mapper returns its list of `(word, 1)` pairs.

In [5]:
def map_task(path):
    return mapper(path.read_text(encoding="utf-8"))

with ThreadPoolExecutor(max_workers=len(splits)) as pool:
    all_pairs = list(pool.map(map_task, splits))
    print(f"mapper emitted {sum(len(p) for p in all_pairs)} pairs")

for path, pairs in zip(splits, all_pairs):
    print(f"{path.name}: mapper emitted {len(pairs)} pairs")

mapper emitted 235 pairs
book_01.txt: mapper emitted 49 pairs
book_02.txt: mapper emitted 48 pairs
book_03.txt: mapper emitted 45 pairs
book_04.txt: mapper emitted 46 pairs
book_05.txt: mapper emitted 47 pairs


Now apply the combiner to each mapper's pairs to get local subtotals (one dictionary per split).

In [6]:
mapper_outputs = [combiner(pairs) for pairs in all_pairs]

## 5. Partition: choose a reducer for each word

The **partitioner** decides which of the 2 reducers receives a word. The rule must be
the same everywhere, so the same word always goes to the same reducer.



In [7]:
def partitioner(word, num_reducers=NUM_REDUCERS):
    return sum(ord(c) for c in word) % num_reducers

for w in ["the", "cat", "dog", "book"]:
    print(w, "-> reducer", partitioner(w))

print(ord("a"))    

the -> reducer 1
cat -> reducer 0
dog -> reducer 0
book -> reducer 1
97


## 6. Shuffle and sort

**Shuffle:** every mapper's subtotals are sent to the reducer chosen by the partitioner.
Each reducer collects a list of values per word, e.g. `"the": [3500, 2900, 4100, 3000, 3300]`.

**Sort:** each reducer orders its words alphabetically.

In [8]:
# one dictionary per reducer
reducer_inputs = [defaultdict(list) for _ in range(NUM_REDUCERS)]

for local_counts in mapper_outputs:              
    for word, subtotal in local_counts.items():
        r = partitioner(word)
        reducer_inputs[r][word].append(subtotal)

# SORT: alphabetical order of words
sorted_inputs = [sorted(r.items()) for r in reducer_inputs]

for i, items in enumerate(sorted_inputs):
    print(f"Reducer {i} received {len(items)} distinct words")

Reducer 0 received 91 distinct words
Reducer 1 received 91 distinct words


## 7. Reduce and write one output file per reducer

The **reducer** receives a word with its list of subtotals and sums them into the final count.
Each reducer writes its own file: `reducer_0.txt` and `reducer_1.txt`.

In [9]:
def reducer(values):
    return sum(values)

OUTPUT_DIR.mkdir(exist_ok=True)

for i, items in enumerate(sorted_inputs):
    out_path = OUTPUT_DIR / f"reducer_{i}.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        for word, values in items:
            f.write(f"{word}\t{reducer(values)}\n")
    print("wrote", out_path, f"({len(items)} words)")

wrote output\reducer_0.txt (91 words)
wrote output\reducer_1.txt (91 words)


## output of both

In [10]:
rows = []

for i in range(NUM_REDUCERS):
    with open(OUTPUT_DIR / f"reducer_{i}.txt", encoding="utf-8") as reducer:
        rows.extend(reducer.readlines())

rows.sort(key=lambda line: line.split("\t", 1)[0])

output_path = OUTPUT_DIR / "output.txt"
with open(output_path, "w", encoding="utf-8") as combined:
    combined.writelines(rows)

print(f"Wrote {output_path.resolve()}")

Wrote C:\Users\mohamed belal\Desktop\Mapreduce\Big-Data-Analytics\mapreduce-word-count\output\output.txt


## 8. Check the results

First 10 lines of each output file, and a sanity check that the grand total equals the total number of words in all books.

In [11]:
for i in range(NUM_REDUCERS):
    print(f"--- reducer_{i}.txt (first 10 lines) ---")
    with open(OUTPUT_DIR / f"reducer_{i}.txt", encoding="utf-8") as f:
        for _, line in zip(range(10), f):
            print(line.rstrip())

total_from_output = sum(
    int(line.split("\t")[1])
    for i in range(NUM_REDUCERS)
    for line in open(OUTPUT_DIR / f"reducer_{i}.txt", encoding="utf-8")
)
total_words = sum(len(tokenize(p.read_text(encoding="utf-8"))) for p in splits)
print("\nTotal words in output:", total_from_output)
print("Total words in books :", total_words)
print("Match:", total_from_output == total_words)

--- reducer_0.txt (first 10 lines) ---
advanced	1
aggregation	1
analysis	2
analytics	2
are	1
behavior	1
big	1
can	5
careful	1
combiners	1
--- reducer_1.txt (first 10 lines) ---
a	1
accuracy	1
across	1
aggregate	1
all	1
an	1
and	12
another	1
before	2
begin	1

Total words in output: 235
Total words in books : 235
Match: True



# Top-K Words

The reducer output files are alphabetical. To find the **most frequent** words we read both
reducer files, merge them, and sort by count (highest first).

In [12]:
K = 10

all_counts = []
for i in range(NUM_REDUCERS):
    with open(OUTPUT_DIR / f"reducer_{i}.txt", encoding="utf-8") as f:
        for line in f:
            word, count = line.rstrip("\n").split("\t")
            all_counts.append((word, int(count)))

top_k = sorted(all_counts, key=lambda wc: (-wc[1], wc[0]))[:K]

print(f"Top {K} words:")
for rank, (word, count) in enumerate(top_k, start=1):
    print(f"{rank:>2}. {word:<15} {count}")

Top 10 words:
 1. and             12
 2. can             5
 3. for             4
 4. the             4
 5. words           4
 6. mapper          3
 7. reducer         3
 8. students        3
 9. with            3
10. analysis        2
